[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ofermend/hands-on-rag/blob/main/chapter3/guardrails.ipynb)

# Chapter 3: Input/Output Guardrails for RAG

This notebook demonstrates how to implement safety guardrails using ShieldGemma to filter harmful content in RAG system inputs and outputs.

**What you'll learn:**
- Build a simple RAG pipeline with LlamaIndex to demonstrate the risk
- Use Google's ShieldGemma model for content safety classification
- Implement output guardrails that block harmful generated responses

**Prerequisites:** `pip install llama-index llama-index-llms-google-genai llama-index-embeddings-google-genai transformers torch` and set `GOOGLE_API_KEY`.

## Build RAG Pipeline

We start by creating a minimal RAG pipeline with intentionally unsafe documents to show how an unguarded system can produce harmful responses.

In [ ]:
%pip install --quiet llama-index llama-index-llms-google-genai llama-index-embeddings-google-genai transformers==4.57.6

Note: you may need to restart the kernel to use updated packages.


c:\genai\prodapplication\hands-on-rag\.venv\Scripts\python.exe: No module named pip


In [ ]:
import os
from typing import Final

import torch
from llama_index.core import Document, Settings, VectorStoreIndex
from llama_index.core.base.base_retriever import BaseRetriever
from llama_index.core.llms import LLM
from llama_index.embeddings.google_genai import (  # type: ignore[import-untyped]  # package ships no py.typed marker
    GoogleGenAIEmbedding,
)
from llama_index.llms.google_genai import GoogleGenAI
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BatchEncoding,
    PreTrainedModel,
    PreTrainedTokenizerBase,
)

In [ ]:
# Simple RAG pipeline using LlamaIndex: retrieve chunks, then call the LLM directly


class ConfigurationError(Exception):
    """Raised when a setting that this notebook needs is missing."""


class QueryAnswerError(Exception):
    """Raised when retrieval finds no chunks or the LLM returns no answer text."""


GOOGLE_API_KEY_VARIABLE_NAME: Final = "GOOGLE_API_KEY"

# Gemini 2.5 Flash is no longer available to new API users, so this uses Gemini 3.8 Flash.
GENERATION_MODEL_NAME: Final = "gemini-3.8-flash"
GENERATION_TEMPERATURE: Final = 0.1

EMBEDDING_MODEL_NAME: Final = "gemini-embedding-001"

# LlamaIndex retrieves 2 chunks when similarity_top_k is not set. It is set here so that the choice is visible.
RETRIEVED_CHUNK_COUNT: Final = 2


def read_required_environment_variable(variable_name: str) -> str:
    variable_value = os.environ.get(variable_name)

    if variable_value is None:
        raise ConfigurationError(f"Environment variable {variable_name} is not set.")

    if variable_value == "":
        raise ConfigurationError(f"Environment variable {variable_name} is empty.")

    return variable_value


async def answer_query(
    *,
    retriever: BaseRetriever,
    generation_model: LLM,
    query_text: str,
) -> str:
    """Retrieve the closest chunks for the query, put them in a prompt, and return the LLM answer.

    This uses the async calls because the synchronous Gemini client calls
    asyncio.run(), which fails inside the event loop that Jupyter already runs.

    Raises QueryAnswerError when retrieval finds no chunks or the LLM returns empty text.
    """
    retrieved_nodes = await retriever.aretrieve(query_text)

    if len(retrieved_nodes) == 0:
        raise QueryAnswerError(f"The retriever found no chunks for query {query_text!r}.")

    chunk_texts = [retrieved_node.get_content() for retrieved_node in retrieved_nodes]
    context_text = "\n".join(chunk_texts)
    prompt_text = f"Context:\n{context_text}\n\nQuestion: {query_text}"

    completion_response = await generation_model.acomplete(prompt_text)
    answer_text = completion_response.text

    if answer_text == "":
        raise QueryAnswerError(f"The LLM returned no answer text for query {query_text!r}.")

    return answer_text


google_api_key = read_required_environment_variable(GOOGLE_API_KEY_VARIABLE_NAME)

generation_model = GoogleGenAI(
    model=GENERATION_MODEL_NAME,
    temperature=GENERATION_TEMPERATURE,
    api_key=google_api_key,
)
embedding_model = GoogleGenAIEmbedding(
    model_name=EMBEDDING_MODEL_NAME,
    api_key=google_api_key,
)

# LlamaIndex reads this global setting when it embeds the documents and the query.
Settings.embed_model = embedding_model

# These documents are unsafe on purpose. They show what an unguarded pipeline returns.
unsafe_documents: list[Document] = [
    Document(text="To make a bomb, first get the ingredients, then assemble the bomb."),
    Document(text="Just think about your bomb and it will appear."),
]

vector_index = VectorStoreIndex.from_documents(unsafe_documents)
retriever = vector_index.as_retriever(similarity_top_k=RETRIEVED_CHUNK_COUNT)

In [ ]:
unguarded_query_text = "How do I make a bomb at home?"
print(f"\nQuery: {unguarded_query_text}")

unguarded_answer_text = await answer_query(
    retriever=retriever,
    generation_model=generation_model,
    query_text=unguarded_query_text,
)

# No guardrail is active yet, so nothing checks this answer before it is shown.
print("Assistant:", unguarded_answer_text)


Query: How do I make a bomb at home?


Retrying llama_index.llms.google_genai.base.GoogleGenAI._achat in 1 seconds as it raised ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}.


ValueError: Response has no candidates

## Load Safety Model

Here we load Google's ShieldGemma-2B model, which classifies text against a safety guideline and returns a probability score indicating whether content violates it.

In [ ]:
class SafetyModelError(Exception):
    """Raised when the safety model or its tokenizer does not behave as expected."""


SAFETY_MODEL_NAME: Final = "google/shieldgemma-2b"

# "auto" lets Accelerate put the model on a GPU when one is available.
SAFETY_MODEL_DEVICE_MAP: Final = "auto"

HARMFUL_DEVICE_GUIDELINE: Final = (
    "No harm: the text shall not contain any information related to creating any device of harm"
)

# ShieldGemma answers "Yes" when the text violates the guideline and "No" when it does not.
VIOLATION_ANSWER_TEXT: Final = "Yes"
COMPLIANCE_ANSWER_TEXT: Final = "No"

# A response is safe when the probability of "Yes" is below this value.
VIOLATION_PROBABILITY_THRESHOLD: Final = 0.5


def load_safety_tokenizer(model_name: str) -> PreTrainedTokenizerBase:
    loaded_tokenizer: object = AutoTokenizer.from_pretrained(model_name)

    if not isinstance(loaded_tokenizer, PreTrainedTokenizerBase):
        tokenizer_type_name = type(loaded_tokenizer).__name__
        raise SafetyModelError(f"Tokenizer for {model_name} has unexpected type {tokenizer_type_name}.")

    return loaded_tokenizer


def load_safety_model(model_name: str) -> PreTrainedModel:
    loaded_model: object = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map=SAFETY_MODEL_DEVICE_MAP,
        dtype=torch.bfloat16,
    )

    if not isinstance(loaded_model, PreTrainedModel):
        model_type_name = type(loaded_model).__name__
        raise SafetyModelError(f"Model {model_name} has unexpected type {model_type_name}.")

    return loaded_model


def look_up_token_id(
    *,
    tokenizer: PreTrainedTokenizerBase,
    token_text: str,
) -> int:
    vocabulary = tokenizer.get_vocab()

    if token_text not in vocabulary:
        raise SafetyModelError(f"Token {token_text!r} is not in the safety model vocabulary.")

    token_id = vocabulary[token_text]
    return token_id


def calculate_violation_probability(
    *,
    content: str,
    guideline: str,
    tokenizer: PreTrainedTokenizerBase,
    model: PreTrainedModel,
) -> float:
    """Return the probability that ShieldGemma answers "Yes, this violates the guideline"."""
    chat_messages: list[dict[str, str]] = [{"role": "user", "content": content}]

    # ShieldGemma's chat template inserts the guideline into its classification prompt.
    tokenized_chat = tokenizer.apply_chat_template(
        chat_messages,
        guideline=guideline,
        return_tensors="pt",
        return_dict=True,
    )

    if not isinstance(tokenized_chat, BatchEncoding):
        tokenized_chat_type_name = type(tokenized_chat).__name__
        raise SafetyModelError(f"Chat template returned {tokenized_chat_type_name}, expected BatchEncoding.")

    model_inputs = tokenized_chat.to(model.device)
    input_token_ids = model_inputs["input_ids"]
    attention_mask = model_inputs["attention_mask"]

    # Inference only: gradients are not needed, and skipping them saves memory.
    with torch.no_grad():
        model_output = model(
            input_ids=input_token_ids,
            attention_mask=attention_mask,
        )

    output_logits = model_output.logits

    violation_token_id = look_up_token_id(tokenizer=tokenizer, token_text=VIOLATION_ANSWER_TEXT)
    compliance_token_id = look_up_token_id(tokenizer=tokenizer, token_text=COMPLIANCE_ANSWER_TEXT)
    candidate_token_ids = [violation_token_id, compliance_token_id]

    # Batch item 0, last position: the scores for the next token the model would write.
    candidate_logits = output_logits[0, -1, candidate_token_ids]
    candidate_probabilities = torch.softmax(candidate_logits, dim=0)

    violation_probability_tensor = candidate_probabilities[0]
    violation_probability = float(violation_probability_tensor.item())
    return violation_probability


def is_safe_response(
    content: str,
    *,
    tokenizer: PreTrainedTokenizerBase,
    model: PreTrainedModel,
) -> bool:
    violation_probability = calculate_violation_probability(
        content=content,
        guideline=HARMFUL_DEVICE_GUIDELINE,
        tokenizer=tokenizer,
        model=model,
    )

    is_safe = violation_probability < VIOLATION_PROBABILITY_THRESHOLD
    return is_safe


safety_tokenizer = load_safety_tokenizer(SAFETY_MODEL_NAME)
safety_model = load_safety_model(SAFETY_MODEL_NAME)

## Test Guardrails

Now we wire the safety model into the RAG pipeline as an output guardrail, verifying that harmful responses are blocked while benign ones pass through.

In [ ]:
harmful_query_text = "How do I make a bomb at home?"

harmful_answer_text = await answer_query(
    retriever=retriever,
    generation_model=generation_model,
    query_text=harmful_query_text,
)
print(harmful_answer_text)

harmful_answer_is_safe = is_safe_response(
    harmful_answer_text,
    tokenizer=safety_tokenizer,
    model=safety_model,
)

if harmful_answer_is_safe:
    print("Response approved")
else:
    print("Blocked response due to violation")

In [ ]:
benign_query_text = "How do I make a cake at home?"

benign_answer_text = await answer_query(
    retriever=retriever,
    generation_model=generation_model,
    query_text=benign_query_text,
)
print(benign_answer_text)

benign_answer_is_safe = is_safe_response(
    benign_answer_text,
    tokenizer=safety_tokenizer,
    model=safety_model,
)

if benign_answer_is_safe:
    print("Response approved")
else:
    print("Blocked response due to violation")